In [1]:
# 데이터 불러오기
import pandas as pd

file_path = "/content/drive/MyDrive/JeonseGuard/실거래가/매매/연립다세대/202504_연립다세대_매매_실거래가.csv" # CSV 파일 경로 지정
df = pd.read_csv(file_path, encoding='cp949') # CP949 인코딩

In [2]:
# 상위 5개 확인
df.head()

,NO,시군구,번지,본번,부번,건물명,전용면적(㎡),대지권면적(㎡),계약년월,계약일,...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
0,1,서울특별시 강남구 개포동,1239-12,1239,12,더푸른빌,43.50,27.1250,202504,30,...,6,개인,개인,2016,개포로15길 3-15,-,중개거래,서울 강남구,-,다세대
1,2,인천광역시 부평구 십정동,178-9,178,9,금호주택(178-9),47.20,19.1500,202504,30,...,4,개인,개인,2001,신촌로 23,-,중개거래,인천 부평구,-,다세대
2,3,경기도 광주시 퇴촌면 도수리,564-1,564,1,노블레스106동,71.87,116.0440,202504,30,...,1,개인,개인,2011,도지울길12번길 18,-,중개거래,경기 광주시,-,다세대
3,4,경기도 광주시 쌍령동,262-37,262,37,쌍령하이츠505동,59.73,42.7087,202504,30,...,2,개인,개인,2008,경충대로 1514-12,-,중개거래,경기 광주시,-,다세대
4,5,경기도 광주시 신현동,421-15,421,15,블레스힐(102동),68.29,69.8750,202504,30,...,3,개인,개인,2018,문형산길157번길 37-3,-,중개거래,경기 광주시,25.05.15,다세대


In [3]:
# 컬럼명 확인
print(df.columns)

Index(['NO', '시군구', '번지', '본번', '부번', '건물명', '전용면적(㎡)', '대지권면적(㎡)', '계약년월',
       '계약일', '거래금액(만원)', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일', '거래유형',
       '중개사소재지', '등기일자', '주택유형'],
      dtype='object')


In [4]:
# 특정 컬럼 값만 확인
df["거래금액(만원)"].head()

,거래금액(만원)
0,"82,500"
1,"21,650"
2,"15,400"
3,"17,900"
4,"25,000"


In [5]:
# 변경 매핑 딕셔너리 정의
renamed_columns = {
    "시군구": "address",
    "본번": "bun",
    "부번": "ji",
    "층": "floor",
    "전용면적(㎡)": "area",
    "계약년월": "contract_year_month",
    "거래금액(만원)": "price",
    "주택유형": "housing_type"
}

In [6]:
# 매핑에 해당하는 컬럼만 필터링
df = df.rename(columns=renamed_columns)
sale_df = df[list(renamed_columns.values())].copy()
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,서울특별시 강남구 개포동,1239,12,6,43.50,202504,"82,500",다세대
1,인천광역시 부평구 십정동,178,9,4,47.20,202504,"21,650",다세대
2,경기도 광주시 퇴촌면 도수리,564,1,1,71.87,202504,"15,400",다세대
3,경기도 광주시 쌍령동,262,37,2,59.73,202504,"17,900",다세대
4,경기도 광주시 신현동,421,15,3,68.29,202504,"25,000",다세대


In [7]:
# 쉼표 제거 및 문자열을 정수로 변환
sale_df["price"] = (
    sale_df["price"]
    .astype(str) # 문자열로 변환 (안전)
    .str.replace(",", "") # 쉼표 제거
    .astype(int) # 정수형으로 변환
    * 10000 # 만원 → 원 변환
)

In [8]:
# 변환된 price 컬럼 확인
sale_df[["price"]].head()

,price
0,825000000
1,216500000
2,154000000
3,179000000
4,250000000


In [9]:
# price 컬럼의 값을 쉼표가 포함된 문자열로 변환
sale_df["price"] = sale_df["price"].apply(lambda x: format(x, ","))
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,서울특별시 강남구 개포동,1239,12,6,43.50,202504,"825,000,000",다세대
1,인천광역시 부평구 십정동,178,9,4,47.20,202504,"216,500,000",다세대
2,경기도 광주시 퇴촌면 도수리,564,1,1,71.87,202504,"154,000,000",다세대
3,경기도 광주시 쌍령동,262,37,2,59.73,202504,"179,000,000",다세대
4,경기도 광주시 신현동,421,15,3,68.29,202504,"250,000,000",다세대


In [10]:
# 결측치 확인
missing_counts = sale_df.isnull().sum()
print("📌 결측치가 있는 컬럼: ", missing_counts[missing_counts > 0])

📌 결측치가 있는 컬럼:  Series([], dtype: int64)


In [11]:
# 전체 중복된 행의 수 확인
duplicate_count = sale_df.duplicated().sum()
print(f"📌 중복된 행의 수: {duplicate_count}")

📌 중복된 행의 수: 255


In [12]:
# 중복 제거 (기본: 모든 열 기준, keep='first')
sale_df = sale_df.drop_duplicates().reset_index(drop=True)

In [13]:
# INSERT 구문 생성
insert_header = """INSERT INTO transaction_sale_rowhouse (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES """

In [14]:
# 행별로 SQL 값 문자열 생성
values = []

In [15]:
# 각 행을 values 리스트에 추가
for _, row in sale_df.iterrows():
    values.append(f"('{row['address']}', '{row['bun']}', '{row['ji']}', '{row['floor']}', '{row['area']}', '{row['contract_year_month']}', '{row['price']}', '{row['housing_type']}', NOW(), NOW())")

In [16]:
# INSERT 구문 상위 5개만 출력
preview_sql = insert_header + ",\n       ".join(values[:5]) + ";"
print(preview_sql)

INSERT INTO transaction_sale_rowhouse (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES ('서울특별시 강남구 개포동', '1239', '12', '6', '43.5', '202504', '825,000,000', '다세대', NOW(), NOW()),
       ('인천광역시 부평구 십정동', '178', '9', '4', '47.2', '202504', '216,500,000', '다세대', NOW(), NOW()),
       ('경기도 광주시 퇴촌면 도수리', '564', '1', '1', '71.87', '202504', '154,000,000', '다세대', NOW(), NOW()),
       ('경기도 광주시 쌍령동', '262', '37', '2', '59.73', '202504', '179,000,000', '다세대', NOW(), NOW()),
       ('경기도 광주시 신현동', '421', '15', '3', '68.29', '202504', '250,000,000', '다세대', NOW(), NOW());


In [17]:
# INSERT 구문 조립
insert_sql = insert_header + ",\n       ".join(values) + ";"

In [18]:
# 파일 저장
file_name = "V44__insert_transaction_sale_officetel_202504.sql"

with open(file_name, "w", encoding="utf-8") as f:
    f.write(insert_sql)

print(f"{file_name} 파일이 생성되었습니다.")

V44__insert_transaction_sale_officetel_202504.sql 파일이 생성되었습니다.
